---
**题名:** 张量并行

**门类:** tensor-parallelism

**难易:** 中

**所需时辰:** 约三刻

---

## 总论

模型之层若巨，一器不能容，则**张量并行（TP）**者，分权重矩阵于众GPU也。

此篇以实数为例，循序渐进，阐明其理：

1. 按列分矩阵 → **列并行**
2. 按行分矩阵 → **行并行**
3. 二者合为**共轭对**（每块仅一次AllReduce）
4. 施于**MLP**及**自注意力**

### 先修之学
- PyTorch基础（张量运算、矩阵乘法）
- [00-gpu-communication/](../00-gpu-communication/) — 尤须明 **AllReduce**（各器将己数传于众，终使各器皆得诸数之总和）与 **AllGather**（各器将己数传出，终使各器皆得诸数之拼接）

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from mp_tutorial.viz import show_matrix, show_matrices_row, draw_tensor_split, GPU_COLORS
from mp_tutorial.distributed import simulate_allreduce, simulate_allgather, check_gpu_env

check_gpu_env()
torch.manual_seed(42)

%matplotlib inline

---
## 一、最简之例：单一线性层

**线性层**者，神经网络至基之构件也 — 其算 $Y = X \cdot W$，$X$ 为输入，$W$ 为**权重矩阵**（一组可学之数）。训练模型者，即调此权重之值也。

今以小张量始之，使可尽览。

In [ ]:
# 小型线性层：X (2×4) @ W (4×6) = Y (2×6)
X = torch.tensor([[1., 2., 3., 4.],
                   [5., 6., 7., 8.]])

W = torch.tensor([[1., 0., 2., 1., 0., 1.],
                   [0., 1., 1., 0., 2., 0.],
                   [1., 1., 0., 1., 1., 2.],
                   [2., 0., 1., 0., 1., 1.]])

Y = X @ W

fig, axes = plt.subplots(1, 3, figsize=(14, 2.5))
show_matrix(X, ax=axes[0], title="X（输入）", cmap="Blues")
show_matrix(W, ax=axes[1], title="W（权重）", cmap="Greens")
show_matrix(Y, ax=axes[2], title="Y = X @ W（输出）", cmap="YlOrRd")
fig.suptitle("常规线性层 —— 尽在一器之中", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

然若 $W$ 之大，一器不能载，则必**分之**。

---
## 二、列并行：按列分 $W$

将 $W$ 之列均分于众GPU。各器得全量输入 $X$，各以其所分之 $W$ 计算。

In [ ]:
# 将W分为三列块（模拟三GPU）
num_gpus = 3
W_chunks = W.chunk(num_gpus, dim=1)  # 按列分割

# 可视化分割
fig = show_matrices_row(
    W_chunks,
    titles=["W₀", "W₁", "W₂"],
    gpu_labels=["GPU 0", "GPU 1", "GPU 2"],
    suptitle="第一步：按列分W → 各器得二列",
    cmap="Greens"
)
plt.show()

In [ ]:
# 各器计算：Y_i = X @ W_i（无需通传！）
Y_parts = [X @ W_i for W_i in W_chunks]

fig = show_matrices_row(
    Y_parts,
    titles=["Y₀ = X @ W₀", "Y₁ = X @ W₁", "Y₂ = X @ W₂"],
    gpu_labels=["GPU 0", "GPU 1", "GPU 2"],
    suptitle="第二步：各器独行矩阵乘（无需通传！）"
)
plt.show()

In [ ]:
# 拼接以得全量输出
Y_column = torch.cat(Y_parts, dim=-1)

fig, axes = plt.subplots(1, 2, figsize=(12, 2.5))
show_matrix(Y_column, ax=axes[0], title="拼接 Y₀|Y₁|Y₂")
show_matrix(Y,        ax=axes[1], title="原始 Y = X @ W")
fig.suptitle("第三步：拼接所得 → 与单器之果无异",
             fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

assert torch.allclose(Y_column, Y)
print("列并行验证通过！")

**要旨：** 列并行之前向，**无需通传**。各器独算其所分之输出列。

---
## 三、行并行：按行分 $W$

今按行分 $W$。如此则**输入** $X$ 亦须沿末维分之。

异于列并行者，此处各器所产为**部分和**（形与全量输出同）。欲得终果，须行 **AllReduce** — 集体通传之术也，各器将己之部分结果传于众器，终使各器皆得总和。

In [ ]:
# 按行分W、按列分X（末维），此次用二GPU
num_gpus = 2
W_chunks = W.chunk(num_gpus, dim=0)  # 按行分W
X_chunks = X.chunk(num_gpus, dim=1)  # 按列分X

fig, axes = plt.subplots(2, 2, figsize=(10, 5))
show_matrix(X_chunks[0], ax=axes[0, 0], title="X₀（前二列）", gpu_label="GPU 0", cmap="Blues")
show_matrix(W_chunks[0], ax=axes[0, 1], title="W₀（前二行）", gpu_label="GPU 0", cmap="Greens")
show_matrix(X_chunks[1], ax=axes[1, 0], title="X₁（后二列）",  gpu_label="GPU 1", cmap="Blues")
show_matrix(W_chunks[1], ax=axes[1, 1], title="W₁（后二行）",  gpu_label="GPU 1", cmap="Greens")
fig.suptitle("第一步：按行分W，相应分X", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# 各器算得部分之果：Y_i = X_i @ W_i
Y_parts = [X_i @ W_i for X_i, W_i in zip(X_chunks, W_chunks)]

fig = show_matrices_row(
    Y_parts,
    titles=["Y₀ = X₀ @ W₀（部分）", "Y₁ = X₁ @ W₁（部分）"],
    gpu_labels=["GPU 0", "GPU 1"],
    suptitle="第二步：各器算得部分和（形同全量！）"
)
plt.show()

print(f"观之：二部分之形皆为 {Y_parts[0].shape} —— 与全量输出同。")
print("此乃部分和，须相加而合之。")

In [ ]:
# AllReduce：汇总各器之部分结果
Y_allreduced = simulate_allreduce(Y_parts)

fig, axes = plt.subplots(1, 3, figsize=(16, 2.5))
show_matrix(Y_parts[0],      ax=axes[0], title="Y₀（部分）", gpu_label="GPU 0")
show_matrix(Y_parts[1],      ax=axes[1], title="Y₁（部分）", gpu_label="GPU 1")
show_matrix(Y_allreduced[0], ax=axes[2], title="Y₀ + Y₁ = Y (AllReduce)")

# 于图间加"+"与"="
fig.text(0.345, 0.45, "+", fontsize=24, fontweight="bold", ha="center")
fig.text(0.67,  0.45, "→ AllReduce →", fontsize=12, fontweight="bold", ha="center",
         color="#C44E52")
fig.suptitle("第三步：AllReduce合部分和 → 得全量结果", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

assert torch.allclose(Y_allreduced[0], Y)
print("行并行验证通过！")

**要旨：** 行并行须一次 **AllReduce（求和）** 以合部分之果。

---
## 四、两法并观

今以一图并列二策，以明其异。

In [ ]:
def draw_tp_comparison():
    """列并行与行并行之对比图。"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    for ax, mode in [(ax1, "column"), (ax2, "row")]:
        ax.set_xlim(0, 10)
        ax.set_ylim(0, 10)
        ax.axis("off")

        if mode == "column":
            ax.set_title("列并行", fontsize=14, fontweight="bold",
                        color=GPU_COLORS[0])
            steps = [
                (5, 9.0, "X（全量，各器复制）", "#888"),
                (3, 7.0, "X @ W₀", GPU_COLORS[0]),
                (7, 7.0, "X @ W₁", GPU_COLORS[1]),
                (5, 5.0, "无需通传", "#55A868"),
                (5, 3.0, "Cat([Y₀, Y₁]) = Y", "#888"),
            ]
        else:
            ax.set_title("行并行", fontsize=14, fontweight="bold",
                        color=GPU_COLORS[1])
            steps = [
                (5, 9.0, "X 分之 → X₀, X₁", "#888"),
                (3, 7.0, "X₀ @ W₀", GPU_COLORS[0]),
                (7, 7.0, "X₁ @ W₁", GPU_COLORS[1]),
                (5, 5.0, "AllReduce（求和）", "#C44E52"),
                (5, 3.0, "Y₀ + Y₁ = Y", "#888"),
            ]

        for x, y, text, color in steps:
            box = mpatches.FancyBboxPatch(
                (x - 2.5, y - 0.55), 5.0, 1.1,
                boxstyle="round,pad=0.15", facecolor=color, alpha=0.15,
                edgecolor=color, linewidth=2
            )
            ax.add_patch(box)
            ax.text(x, y, text, ha="center", va="center",
                    fontsize=11, fontweight="bold")

        # 箭头
        arrows = [(5, 8.4, 3, 7.6), (5, 8.4, 7, 7.6),
                  (3, 6.4, 5, 5.6), (7, 6.4, 5, 5.6),
                  (5, 4.4, 5, 3.6)]
        for x1, y1, x2, y2 in arrows:
            ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                       arrowprops=dict(arrowstyle="->", lw=1.5, color="#555"))

    fig.tight_layout()
    return fig

fig = draw_tp_comparison()
plt.show()

| | 列并行 | 行并行 |
|---|---|---|
| **W之分法** | 按列 | 按行 |
| **X** | 全量复制 | 沿末维分之 |
| **各器输出** | Y之列切片 | 部分和（形同Y） |
| **通传** | 无（或AllGather） | **AllReduce**（求和） |

---
## 五、共轭对：列行合用于MLP

Transformer之 **MLP（多层感知机）块**者，各Transformer层内之小型前馈网络也 — 有二线性层，中间夹一激活函数：

$$\text{MLP}(X) = \text{GeLU}(X \cdot W_1) \cdot W_2$$

**GeLU**（高斯误差线性单元）者，激活函数也 — 类ReLU而更平滑。引非线性，使网络能学复杂之模式。于TP而言要者：GeLU逐元素而施，故各器可独立施之于己之切片。

Megatron之妙法：以**列并行**施于 $W_1$，继以**行并行**施于 $W_2$。列并行之输出，恰已按行并行所需之形分割 —— **二层之间无需通传！**

今以实数验之。

In [ ]:
# MLP：隐层=4 → 中间层=6 → 隐层=4，三GPU并行
torch.manual_seed(7)
X  = (torch.randn(2, 4) * 2).round() / 2   # 取整数以便观察
W1 = (torch.randn(4, 6) * 2).round() / 2
W2 = (torch.randn(6, 4) * 2).round() / 2

# 参考结果
Y_ref = F.gelu(X @ W1) @ W2

print("形制：X(2×4) → W1(4×6) → GeLU → W2(6×4) → Y(2×4)")
print(f"三GPU：各得W1之二列、W2之二行")

In [ ]:
num_gpus = 3

# 第一步：列并行分W1
W1_chunks = W1.chunk(num_gpus, dim=1)

fig, axes = plt.subplots(1, 4, figsize=(16, 3),
                         gridspec_kw={"width_ratios": [4, 2, 2, 2]})
show_matrix(W1, ax=axes[0], title="W₁（全量）", cmap="Greens")
for i, chunk in enumerate(W1_chunks):
    show_matrix(chunk, ax=axes[i+1], title=f"W₁ᵢ", gpu_label=f"GPU {i}", cmap="Greens")
fig.suptitle("MLP第一步：按列分W₁", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# 第二步：各器算 X @ W1_i 后施GeLU
H_parts = [F.gelu(X @ W1_i) for W1_i in W1_chunks]

fig = show_matrices_row(
    H_parts,
    titles=["GeLU(X @ W₁₀)", "GeLU(X @ W₁₁)", "GeLU(X @ W₁₂)"],
    gpu_labels=["GPU 0", "GPU 1", "GPU 2"],
    suptitle="MLP第二步：各器独行矩阵乘与GeLU（无需通传）",
    cmap="Purples"
)
plt.show()

print("各器得(2×2)隐层激活 —— 乃中间层之切片也。")

In [ ]:
# 第三步：行并行分W2 —— 按行分
# 列并行之输出恰已按所需之形分割！
W2_chunks = W2.chunk(num_gpus, dim=0)

fig, axes = plt.subplots(1, 4, figsize=(16, 3),
                         gridspec_kw={"width_ratios": [4, 2, 2, 2]})
show_matrix(W2, ax=axes[0], title="W₂（全量）", cmap="Greens")
for i, chunk in enumerate(W2_chunks):
    show_matrix(chunk, ax=axes[i+1], title=f"W₂ᵢ", gpu_label=f"GPU {i}", cmap="Greens")
fig.suptitle("MLP第三步：按行分W₂（恰与列分相配！）",
             fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# 第四步：各器算 H_i @ W2_i → 部分输出（形同最终Y）
Y_parts = [H_i @ W2_i for H_i, W2_i in zip(H_parts, W2_chunks)]

fig = show_matrices_row(
    Y_parts,
    titles=["H₀ @ W₂₀", "H₁ @ W₂₁", "H₂ @ W₂₂"],
    gpu_labels=["GPU 0", "GPU 1", "GPU 2"],
    suptitle="MLP第四步：各器得输出之部分和"
)
plt.show()

print("三者皆为(2×4) —— 与最终输出同形。须求和以合之。")

In [ ]:
# 第五步：AllReduce —— 整个MLP中唯一之通传！
Y_final = simulate_allreduce(Y_parts)

fig, axes = plt.subplots(1, 2, figsize=(10, 2.5))
show_matrix(Y_final[0], ax=axes[0], title="AllReduce之果")
show_matrix(Y_ref,      ax=axes[1], title="参考（单器MLP）")
fig.suptitle("MLP第五步：AllReduce合三部分 → 与参考吻合",
             fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

assert torch.allclose(Y_final[0], Y_ref, atol=1e-5)
print("张量并行MLP验证通过！整块MLP仅需一次AllReduce。")

### 何以能行？

列并行之输出，天然可为行并行之输入：

| 步骤 | 所行之事 | 通传 |
|------|----------|------|
| W₁列并行 | 各器得W₁之列，以全量X计算 | 无 |
| GeLU | 各器独施于己之切片 | 无 |
| W₂行并行 | 各器之切片恰为行并行所需之输入 | 无 |
| AllReduce | 合部分输出以得最终Y | **一次AllReduce** |

此即**共轭对** —— Megatron式张量并行之基石也。

> **难度渐升：** 下节以 TP 施于自注意力，涉 Q/K/V 投射与多头之分。若于注意力机制尚生，但记要旨：注意力头各自独立，故自然分于众 GPU，无需额外通传。

---
## 六、自注意力：同法异层

**多头自注意力**者，Transformer层之另一要件也（与MLP并列）。使模型能览序列中诸位，判其相关者。

其法：以不同权重矩阵将输入投射为三组向量 — **Q（查询）**、**K（键）**、**V（值）**。位间之注意力分数以 $\text{softmax}(QK^T / \sqrt{d})$ 算之，而后以此加权值 $V$。

"多头"者，以不同之学习投影（名曰**头**）并行施此术也。各头可关注不同之模式。诸头独立，故天然宜于张量并行 — 各器分掌数头。

In [ ]:
def draw_attention_tp(num_heads=8, num_gpus=4):
    """注意力头分布于各GPU之示意图。"""
    heads_per_gpu = num_heads // num_gpus

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.set_xlim(-0.5, num_heads + 3)
    ax.set_ylim(-1, 4.5)
    ax.axis("off")

    # 以色框绘各头
    for h in range(num_heads):
        gpu = h // heads_per_gpu
        color = GPU_COLORS[gpu % len(GPU_COLORS)]
        rect = mpatches.FancyBboxPatch(
            (h, 2), 0.85, 1.5,
            boxstyle="round,pad=0.1", facecolor=color, alpha=0.7,
            edgecolor="white", linewidth=2
        )
        ax.add_patch(rect)
        ax.text(h + 0.42, 2.75, f"H{h}", ha="center", va="center",
                fontsize=11, fontweight="bold", color="white")

    # GPU标识
    for g in range(num_gpus):
        start = g * heads_per_gpu
        mid = start + heads_per_gpu / 2 - 0.08
        color = GPU_COLORS[g % len(GPU_COLORS)]
        ax.text(mid, 1.3, f"GPU {g}", ha="center", va="center",
                fontsize=10, fontweight="bold", color="white",
                bbox=dict(boxstyle="round,pad=0.3", fc=color, ec="none"))
        # 括弧
        ax.plot([start, start + heads_per_gpu - 0.15],
                [1.7, 1.7], color=color, linewidth=2)

    # 标注
    ax.text(num_heads / 2 - 0.08, 4.1,
            "Q、K、V投射：列并行（各器分得其头之权重）",
            ha="center", fontsize=11, fontstyle="italic")
    ax.text(num_heads / 2 - 0.08, 0.3,
            "输出投射：行并行 → AllReduce",
            ha="center", fontsize=11, fontstyle="italic", color="#C44E52")

    ax.set_title(f"{num_heads}注意力头分于{num_gpus}GPU",
                fontsize=13, fontweight="bold")
    fig.tight_layout()
    return fig

fig = draw_attention_tp(num_heads=8, num_gpus=4)
plt.show()

In [ ]:
# 模拟：4头、隐层=8、头维=2、2 GPU
torch.manual_seed(0)
num_heads, head_dim, hidden = 4, 2, 8
num_gpus = 2
seq_len = 3

X = (torch.randn(seq_len, hidden) * 2).round() / 2

# Q、K、V投射权重（全量）：(隐层, 头数 * 头维)
Wq = torch.randn(hidden, num_heads * head_dim)
Wk = torch.randn(hidden, num_heads * head_dim)
Wv = torch.randn(hidden, num_heads * head_dim)
Wo = torch.randn(num_heads * head_dim, hidden)  # 输出投射

# --- 参考：单器注意力 ---
Q = X @ Wq  # (序列长, 头数*头维)
K = X @ Wk
V = X @ Wv

# 变形为 (头数, 序列长, 头维) 以行注意力计算
Q_heads = Q.view(seq_len, num_heads, head_dim).permute(1, 0, 2)
K_heads = K.view(seq_len, num_heads, head_dim).permute(1, 0, 2)
V_heads = V.view(seq_len, num_heads, head_dim).permute(1, 0, 2)

attn = torch.softmax(Q_heads @ K_heads.transpose(-2, -1) / head_dim**0.5, dim=-1)
attn_out = (attn @ V_heads).permute(1, 0, 2).reshape(seq_len, num_heads * head_dim)
Y_ref = attn_out @ Wo
print(f"参考注意力输出之形：{Y_ref.shape}")

In [ ]:
# --- 张量并行注意力：分头于各GPU ---
heads_per_gpu = num_heads // num_gpus  # 每器二头

# 列并行：按列分Wq、Wk、Wv（各器得其头之权重）
Wq_chunks = Wq.chunk(num_gpus, dim=1)
Wk_chunks = Wk.chunk(num_gpus, dim=1)
Wv_chunks = Wv.chunk(num_gpus, dim=1)
# 行并行：按行分Wo
Wo_chunks = Wo.chunk(num_gpus, dim=0)

Y_parts = []
for g in range(num_gpus):
    # 各器算其所分之头
    Q_g = X @ Wq_chunks[g]  # (序列长, 每器头数 * 头维)
    K_g = X @ Wk_chunks[g]
    V_g = X @ Wv_chunks[g]

    Q_h = Q_g.view(seq_len, heads_per_gpu, head_dim).permute(1, 0, 2)
    K_h = K_g.view(seq_len, heads_per_gpu, head_dim).permute(1, 0, 2)
    V_h = V_g.view(seq_len, heads_per_gpu, head_dim).permute(1, 0, 2)

    attn_g = torch.softmax(Q_h @ K_h.transpose(-2, -1) / head_dim**0.5, dim=-1)
    out_g = (attn_g @ V_h).permute(1, 0, 2).reshape(seq_len, heads_per_gpu * head_dim)

    # 行并行输出投射 → 部分结果
    Y_parts.append(out_g @ Wo_chunks[g])

fig = show_matrices_row(
    Y_parts,
    titles=["头0-1 → 部分Y", "头2-3 → 部分Y"],
    gpu_labels=["GPU 0", "GPU 1"],
    suptitle="各器处理其注意力头 → 部分输出"
)
plt.show()

In [ ]:
# AllReduce以得最终结果
Y_tp = simulate_allreduce(Y_parts)

fig, axes = plt.subplots(1, 2, figsize=(10, 2.5))
show_matrix(Y_tp[0], ax=axes[0], title="张量并行注意力（AllReduce后）")
show_matrix(Y_ref,   ax=axes[1], title="参考（单器）")
fig.suptitle("AllReduce → 与单器注意力吻合", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

assert torch.allclose(Y_tp[0], Y_ref, atol=1e-5)
print("张量并行注意力验证通过！同理：列并行 → 本地计算 → 行并行 → AllReduce")

---
## 七、总览全局：一Transformer层

每一Transformer层含注意力块与MLP块。以张量并行行之，各需恰好**一次AllReduce**。

In [ ]:
def draw_transformer_tp_flow():
    """一张量并行Transformer层之通传格局。"""
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 12)
    ax.axis("off")

    blocks = [
        # (x, y, w, h, 文字, 颜色, 通传标注)
        (5, 11,  8, 0.8, "输入X（各器复制）", "#888", None),
        (5, 9.5, 8, 0.8, "Q,K,V：列并行", GPU_COLORS[0], "无通传"),
        (5, 8.0, 8, 0.8, "注意力（各器本地）", GPU_COLORS[2], "无通传"),
        (5, 6.5, 8, 0.8, "输出投射：行并行", GPU_COLORS[1], None),
        (5, 5.0, 8, 0.8, "★ AllReduce 其一", "#C44E52", None),
        (5, 3.5, 8, 0.8, "W₁：列并行 + GeLU", GPU_COLORS[0], "无通传"),
        (5, 2.0, 8, 0.8, "W₂：行并行", GPU_COLORS[1], None),
        (5, 0.5, 8, 0.8, "★ AllReduce 其二", "#C44E52", None),
    ]

    section_labels = [
        (9.8, 8.0, "自注\n意力"),
        (9.8, 2.7, "MLP"),
    ]

    for x, y, w, h, text, color, comm in blocks:
        box = mpatches.FancyBboxPatch(
            (x - w/2, y - h/2), w, h,
            boxstyle="round,pad=0.1", facecolor=color, alpha=0.15,
            edgecolor=color, linewidth=2
        )
        ax.add_patch(box)
        fs = 12 if "★" in text else 10
        fw = "bold" if "★" in text else "bold"
        ax.text(x, y, text, ha="center", va="center", fontsize=fs, fontweight=fw)
        if comm:
            ax.text(x + w/2 - 0.2, y, comm, ha="left", va="center",
                    fontsize=8, fontstyle="italic", color="#55A868")

    # 箭头
    for i in range(len(blocks) - 1):
        y1 = blocks[i][1] - blocks[i][3]/2
        y2 = blocks[i+1][1] + blocks[i+1][3]/2
        ax.annotate("", xy=(5, y2), xytext=(5, y1),
                   arrowprops=dict(arrowstyle="->", lw=1.5, color="#555"))

    for x, y, text in section_labels:
        ax.text(x, y, text, ha="center", va="center", fontsize=10,
                fontstyle="italic", color="#666",
                bbox=dict(boxstyle="round", fc="#f0f0f0", ec="#ccc"))

    ax.set_title("一Transformer层之张量并行\n"
                 "通传总量：仅二次AllReduce",
                 fontsize=13, fontweight="bold")
    fig.tight_layout()
    return fig

fig = draw_transformer_tp_flow()
plt.show()

---
## 八、大语言模型之实践

In [ ]:
def draw_tp_in_practice():
    """张量并行何以限于一节点之内。"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # 左：带宽对比
    ax = axes[0]
    labels = ["NVLink\n（节点内）", "InfiniBand\n（节点间）"]
    bw = [600, 50]
    colors = ["#55A868", "#C44E52"]
    bars = ax.barh(labels, bw, color=colors, height=0.5, edgecolor="white")
    for bar, val in zip(bars, bw):
        ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
                f"{val} GB/s", va="center", fontsize=12, fontweight="bold")
    ax.set_xlim(0, 750)
    ax.set_xlabel("每GPU带宽", fontsize=10)
    ax.set_title("张量并行何以限于节点内", fontsize=12, fontweight="bold")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # 右：模型配置
    ax = axes[1]
    ax.axis("off")
    table_data = [
        ["GPT-3",       "175B", "8",  "8× A100, NVLink"],
        ["LLaMA 2 70B", "70B",  "8",  "8× A100-80GB"],
        ["MT-NLG",      "530B", "8",  "+ 35路PP、64路DP"],
        ["PaLM",        "540B", "8",  "TPU主机内8路"],
    ]
    table = ax.table(
        cellText=table_data,
        colLabels=["模型", "参数量", "TP度", "备注"],
        loc="center", cellLoc="center"
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.0, 1.6)
    for (r, c), cell in table.get_celld().items():
        if r == 0:
            cell.set_facecolor("#4C72B0")
            cell.set_text_props(color="white", fontweight="bold")
        else:
            cell.set_facecolor("#f7f7f7" if r % 2 else "white")
    ax.set_title("实践中张量并行：恒为八度", fontsize=12, fontweight="bold")

    fig.tight_layout()
    return fig

fig = draw_tp_in_practice()
plt.show()

张量并行之度几乎恒为**八** —— 限于一节点内以 **NVLink**（NVIDIA之高速GPU间互联，约600 GB/s）相连之GPU。跨节点则用 **InfiniBand**（约50 GB/s）—— 慢十二倍，不堪TP频繁AllReduce之需。

欲扩展至节点之外，须合TP与**流水线并行**（Pipeline Parallelism, PP —— 按层分模型于不同节点）及**数据并行**（Data Parallelism, DP —— 各节点处理不同之数据批次）。

---
## 九、须GPU之实：真正多器张量并行

> **须GPU** —— 于多卡 GPU 机器执行此节（推荐四卡以上）。

In [ ]:
# [须GPU]
# 以torch.distributed + NCCL实现真正多GPU张量并行
# 于远程机器运行：a multi-GPU machine（四GPU）

import torch
import torch.distributed as dist
import torch.multiprocessing as mp
import torch.nn.functional as F
import os

def tp_mlp_worker(rank, world_size, X_shared, W1_shared, W2_shared, results):
    """张量并行MLP中一GPU之工作函数。"""
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "29500"
    dist.init_process_group("nccl", rank=rank, world_size=world_size)

    device = torch.device(f"cuda:{rank}")
    X = X_shared.to(device)

    # 列并行W1
    chunk_size = W1_shared.shape[1] // world_size
    W1_local = W1_shared[:, rank * chunk_size:(rank + 1) * chunk_size].to(device)

    # 行并行W2
    row_chunk = W2_shared.shape[0] // world_size
    W2_local = W2_shared[rank * row_chunk:(rank + 1) * row_chunk, :].to(device)

    # 前向：列并行 → GeLU → 行并行 → AllReduce
    hidden = F.gelu(X @ W1_local)
    output = hidden @ W2_local
    dist.all_reduce(output, op=dist.ReduceOp.SUM)

    if rank == 0:
        results["output"] = output.cpu()
    dist.destroy_process_group()

if torch.cuda.is_available() and torch.cuda.device_count() >= 4:
    world_size = 4
    hidden, intermediate = 64, 256
    X = torch.randn(2, 8, hidden)
    W1 = torch.randn(hidden, intermediate)
    W2 = torch.randn(intermediate, hidden)

    Y_ref = F.gelu(X @ W1) @ W2

    manager = mp.Manager()
    results = manager.dict()
    mp.spawn(tp_mlp_worker, args=(world_size, X, W1, W2, results), nprocs=world_size)

    Y_tp = results["output"]
    print(f"最大误差：{(Y_tp - Y_ref).abs().max().item():.2e}")
    print("真正多GPU张量并行MLP验证通过！")
else:
    print("略过：须四CUDA GPU。请连接远程：a multi-GPU machine")

---
## 十、Megatron-LM典范

Megatron-LM乃张量并行之正宗产用实现。其关键二类如下：

In [ ]:
from mp_tutorial.formatting import code_reference

code_reference(
    code="""
class ColumnParallelLinear(torch.nn.Module):
    \"\"\"Y = XA + b, with A split by columns: A = [A_1, ..., A_p]\"\"\"

    def __init__(self, input_size, output_size, *, gather_output=True, ...):
        self.output_size_per_partition = divide(output_size, world_size)
        self.weight = Parameter(torch.empty(
            self.output_size_per_partition, input_size))  # transposed!

    def forward(self, input_):
        # Identity in fwd, AllReduce in bwd
        input_parallel = copy_to_tensor_model_parallel_region(input_)
        output_parallel = F.linear(input_parallel, self.weight, self.bias)
        if self.gather_output:
            output = gather_from_tensor_model_parallel_region(output_parallel)
        else:
            output = output_parallel  # keep split for next layer
        return output
""",
    source="Megatron-LM",
    filepath="megatron/core/tensor_parallel/layers.py"
)

In [ ]:
code_reference(
    code="""
class RowParallelLinear(torch.nn.Module):
    \"\"\"Y = XA + b, with A split by rows and X by columns\"\"\"

    def __init__(self, input_size, output_size, *, input_is_parallel=False, ...):
        self.input_size_per_partition = divide(input_size, world_size)
        self.weight = Parameter(torch.empty(
            output_size, self.input_size_per_partition))

    def forward(self, input_):
        if self.input_is_parallel:
            input_parallel = input_  # already split from column-parallel
        else:
            input_parallel = scatter_to_tensor_model_parallel_region(input_)
        output_parallel = F.linear(input_parallel, self.weight)
        # AllReduce in fwd, identity in bwd
        output_ = reduce_from_tensor_model_parallel_region(output_parallel)
        return output_ + self.bias if self.bias is not None else output_
""",
    source="Megatron-LM",
    filepath="megatron/core/tensor_parallel/layers.py"
)

`mappings.py`定义通传原语为自定义autograd函数 —— 各为**共轭对**，以保梯度流之正确：

| 前向 | 反向 | 所用之处 |
|------|------|----------|
| 恒等 | AllReduce | `CopyToModelParallelRegion`（列并行输入） |
| AllReduce | 恒等 | `ReduceFromModelParallelRegion`（行并行输出） |
| 分割 | AllGather | `ScatterToModelParallelRegion` |
| AllGather | 分割 | `GatherFromModelParallelRegion` |

---
## 总结与延伸

### 要义

1. **列并行**按列分 $W$ → 前向无需通传
2. **行并行**按行分 $W$ → 须AllReduce以合部分和
3. **共轭对**（列→行）每块仅需**一次AllReduce**
4. 每Transformer层：注意力一次 + MLP一次 = **共二次**
5. 张量并行度约**八**（限于NVLink相连之一节点内）
6. 自注意力天然相宜：各器分掌数**注意力头**

### 延伸阅读

- [Megatron-LM: Training Multi-Billion Parameter Language Models Using Model Parallelism](https://arxiv.org/abs/1909.08053)
- [Efficient Large-Scale Language Model Training on GPU Clusters](https://arxiv.org/abs/2104.04473) — 三维并行（TP + PP + DP）
- [PyTorch Distributed Docs](https://pytorch.org/docs/stable/distributed.html)
- [NVIDIA Megatron-Core](https://github.com/NVIDIA/Megatron-LM)
- 下篇：[03-pipeline-parallelism/](../03-pipeline-parallelism/) — 按层分模型于各GPU